<a href="https://colab.research.google.com/github/ArlenATU/machineLearningProject/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python machine learning project
Denys Kapustianskyi

## Data Preparation
This section loads the campus dataset and prepares it for analysis

In [3]:
import pandas as pd

## Load the Dataset

In [4]:
from google.colab import drive
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

drive.mount('/content/drive')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

df = pd.read_csv('/content/drive/MyDrive/CampusWalkDataset_TemplateIDs.csv', skiprows = 4)

print("Column names\n")
print(df.dtypes)

df = df.dropna(axis=1, how='all')       #do not print out columns with no data
# Fix: Added .copy() to avoid SettingWithCopyWarning
df_cleaned = df.dropna(thresh=5).copy()

df_cleaned['temperature_c'] = (
    df_cleaned['temperature_c']
    .astype(str)
    .str.extract(r'([\d\.]+)')
    .astype(float)
)

total_rows = len(df_cleaned)
##print(f"Total rows with data: {total_rows}")

#print("\nSample rows\n")
#print(df_cleaned)

features_to_cluster = [
    'temperature_c', 'insects_count', 'birds_count',
    'weather', 'sound level', 'foot_traffic',
    'wellbeing_rating', 'stress_rating'
]


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Column names

recorder_id                               object
date                                      object
location                                  object
weather                                   object
temperature_c                             object
sound level                               object
foot_traffic                              object
primary_sound_type_urban                  object
other_sound_type_urban                    object
primary_sound_type_nature                 object
other_sound_type_nature                   object
insects_count                            float64
birds_count                              float64
trees_presence                            object
native_or_wild_plant_presence             object
ornnamental_or_planted_plant_presence     object
moss_or_lichen_presence                   object
wellbeing_rating        

#Keep only the chosen columns and drop any remaining rows with missing values in them

In [12]:
X = df_cleaned[features_to_cluster].dropna()

I will check what columns are left after cleaning

In [15]:
#I need to see if cleaning ran well and it is correct
for col in X.columns:
    print(col)

temperature_c
insects_count
birds_count
weather
sound level
foot_traffic
wellbeing_rating
stress_rating


#Convert text categories into numerical format (One-Hot Encoding)

In [6]:
X_encoded = pd.get_dummies(X)

#Scale the data so that large numbers don't overpower smaller counts (Standardization)

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

Apply K-Means Clustering (Let's group them into 3 clusters for starters)

In [8]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cleaned.loc[X.index, 'Cluster'] = kmeans.fit_predict(X_scaled)

Check mean values of the groups

In [22]:
df_cleaned.groupby('Cluster').mean(numeric_only=True)

,temperature_c,insects_count,birds_count
Cluster,,,
0.0,16.852941,2.794118,1.705882
1.0,16.756098,2.560976,1.390244
2.0,16.830357,2.964286,2.732143


Print results showing each row assigned to its cluster

In [9]:
print("\nSample rows with Cluster assignments:\n")
# Fix: Changed df_clustered to df_cleaned to resolve NameError
display(df_cleaned.loc[X.index, ['location', 'weather', 'sound level', 'stress_rating', 'Cluster']].head(15))


Sample rows with Cluster assignments:



,location,weather,sound level,stress_rating,Cluster
0,L01 MainEntrance,Partly cloudy,Quiet,Stressful,2.0
1,L02 LibraryGreen,Sunny,Moderate,Neutral,1.0
2,L06 HedgingBehindSportsCentre,Partly cloudy,Quiet,Mostly calm,2.0
3,L08 AccessRoadNearSportsCentre,Partly cloudy,Moderate,Neutral,1.0
4,L06 HedgingBehindSportsCentre,Sunny,Moderate,Mostly calm,2.0
5,L01 MainEntrance,Sunny,Moderate,Mostly calm,2.0
6,L04 PathFrontiHub,Windy,Moderate,Very calm,1.0
7,L05 GreenFrontSportsCentre,Sunny,Moderate,Mostly calm,2.0
8,L06 HedgingBehindSportsCentre,Cloudy,Quiet,Very calm,0.0
9,L07 CarParkNearSportsCentre,Sunny,Quiet,Very calm,0.0


make comments